# 03 — Prospectivity Mapping from Inverted Susceptibility

This notebook converts the recovered magnetic susceptibility model into a practical gold targeting map.

Workflow:
1. Load the recovered susceptibility model from `data/processed/`
2. Compute depth-integrated susceptibility for 0–500 m
3. Overlay bedrock geology contacts from OGS shapefile
4. Compute and plot a prospectivity score map
5. Overlay known Great Bear LP and Hinge zone points for validation
6. Interpret whether known zones align with high prospectivity and propose additional targets

This is the final stage of the pipeline: `01_data_prep` -> `02_inversion` -> `03_prospectivity`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Resolve repository root relative to notebooks folder.
REPO_ROOT = Path.cwd().resolve().parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from inversion_setup import build_mesh
from prospectivity import (
    load_bedrock_geology,
    depth_integrated_susceptibility,
    prospectivity_score,
    plot_prospectivity,
)

In [ ]:
# Load outputs from inversion notebook.
processed_dir = REPO_ROOT / "data" / "processed"
obs_path = processed_dir / "red_lake_tmi_obs.npz"
model_path = processed_dir / "susceptibility_sparse.npy"

if not obs_path.exists():
    raise FileNotFoundError(f"Missing observation file: {obs_path}. Run 01_data_prep.ipynb first.")
if not model_path.exists():
    raise FileNotFoundError(f"Missing model file: {model_path}. Run 02_inversion.ipynb first.")

obs = np.load(obs_path)
receiver_locs = obs["receiver_locations"]
x_obs = np.unique(receiver_locs[:, 0])
y_obs = np.unique(receiver_locs[:, 1])

model_sparse = np.load(model_path)
print(f"Loaded sparse model with {model_sparse.size:,} active-cell values")

In [ ]:
# Rebuild mesh using same parameters as notebook 02.
mesh = build_mesh(x_obs, y_obs, core_cell_size=100.0, padding_cells=8, padding_factor=1.5)

# Current workflow assumes all cells were active during inversion.
actind = np.ones(mesh.n_cells, dtype=bool)
if model_sparse.size != int(actind.sum()):
    raise ValueError(
        "Model size does not match active-cell count. "
        "Ensure notebook 02 used same mesh settings."
    )

print("Mesh and active-cell mask ready.")
print("Mesh shape:", mesh.shape_cells)

In [ ]:
# 1) Compute depth-integrated susceptibility map for 0-500 m depth.
susc_map, x_map, y_map = depth_integrated_susceptibility(
    mesh=mesh,
    model=model_sparse,
    actind=actind,
    z_min=0.0,
    z_max=500.0,
)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(
    susc_map.T,
    origin="lower",
    extent=[x_map.min(), x_map.max(), y_map.min(), y_map.max()],
    aspect="auto",
    cmap="viridis",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("Depth-Integrated Susceptibility (SI*m)")
ax.set_title("Depth-Integrated Susceptibility (0-500 m)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

In [ ]:
# 2) Load bedrock geology and overlay contacts.
raw_dir = REPO_ROOT / "data" / "raw"
shp_candidates = sorted(raw_dir.glob("*.shp"))
if not shp_candidates:
    raise FileNotFoundError(
        f"No shapefile found in {raw_dir}. Add OGS bedrock geology shapefile first."
    )

shapefile_path = shp_candidates[0]
study_bbox = (float(x_map.min()), float(y_map.min()), float(x_map.max()), float(y_map.max()))
geology_gdf = load_bedrock_geology(shapefile_path, study_bbox)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(
    susc_map.T,
    origin="lower",
    extent=[x_map.min(), x_map.max(), y_map.min(), y_map.max()],
    aspect="auto",
    cmap="viridis",
    alpha=0.85,
)
geology_gdf.boundary.plot(ax=ax, color="white", linewidth=0.6)
cb = plt.colorbar(im, ax=ax)
cb.set_label("Depth-Integrated Susceptibility (SI*m)")
ax.set_title("Susceptibility Map with Bedrock Geology Contacts")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

print(f"Using geology shapefile: {shapefile_path.name}")
print(f"Clipped geology features: {len(geology_gdf)}")

In [ ]:
# 3) Compute and plot prospectivity score map.
score_map = prospectivity_score(
    susc_map=susc_map,
    x=x_map,
    y=y_map,
    low_susc_threshold=0.001,
)

plot_prospectivity(
    score_map=score_map,
    x=x_map,
    y=y_map,
    deposit_points=None,
    title="Gold Prospectivity Score Map",
)

In [ ]:
# 4) Overlay known Great Bear validation points.
# Coordinates provided are approximate and in lon/lat.
lp_zone = (-94.05, 51.35)     # (lon, lat)
hinge_zone = (-94.02, 51.32)  # (lon, lat)
deposit_points = [lp_zone, hinge_zone]

plot_prospectivity(
    score_map=score_map,
    x=x_map,
    y=y_map,
    deposit_points=deposit_points,
    title="Prospectivity with Great Bear LP/Hinge Validation Points",
)

print("LP zone (lon, lat):", lp_zone)
print("Hinge zone (lon, lat):", hinge_zone)
print(
    "Note: if your map CRS is projected (e.g., UTM), convert these lon/lat points "
    "to the map CRS before strict spatial interpretation."
)

## Interpretation Prompt (Write-up)

Use this cell for your EOSC 454 interpretation.

1. Do the known Great Bear LP and Hinge zones fall within high-prospectivity areas on your map?
2. If yes/no, explain why based on the susceptibility/gradient patterns and geology contacts.
3. Identify **two additional target areas** and justify each target with:
   - approximate map location,
   - local susceptibility character,
   - proximity to mapped contacts or strong boundaries.
4. Briefly discuss uncertainties (e.g., CRS mismatch, mesh resolution, no topography constraints, assumed IGRF).